<a href="https://colab.research.google.com/github/syslians/-/blob/main/%EB%AA%A8%EB%91%90%EC%9D%98%EC%97%B0%EA%B5%AC%EC%86%8CRAG%EA%B8%B0%EB%B0%98ETF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 프로젝트 2 - Weekend 1: RAG 평가 프레임우크와 하이브리드 검색
### 프로젝트: 검색형 RAG 기반 금융 상품(ETF) 추천 시스템
### 학습목표:
### 1. ETF 데이터셋을 구축하고 벡터 스토어에 색인
### 2. Hit Rate, MRR, NDCG등 검색 평가 지표 구현
### 3. BM25 + 벡터 하이브리도 검색으로 성능 개선
### 4. Query Expansion과 Mutlti-Query로 검색 품질 극대화

In [ ]:
pip install -r requirement.txt

In [ ]:
# 환경 설정 및 라이브러리 설치
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu \
    rank_bm25 pandas numpy matplotlib gradio python-dotenv tiktoken

In [ ]:
pip install --upgrade google-colab requests

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from google.colab import drive

drive.mount('/content/drive')

dotenv_path = '/content/drive/MyDrive/Colab Notebooks/.env'

load_dotenv(dotenv_path)

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

client = OpenAI()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("✅ 환경 설정 완료")

In [ ]:
!pip list | grep langchain

# 문제 1: ETF 사용자 페르소나 정의
## ETF 추천시스템의 사용자 페르소나 3가지 (초보, 중급, 고급)를 정의하세요.
### 요구사항:
    - personas 딕셔너리: 키='초보'/'중급'/'전문'
    - 각 페르소나에 description(설명)과 queries(질의 리스트) 포함
    - 각 질의에 quert(질문 텍스트)와 expected_category(기대 카테고리) 포함
    - project2_data/query_set.json 으로 저장

In [ ]:
import os

# Create the directory if it doesn't exist
output_dir = "project2_data"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"✅ Directory '{output_dir}' created.")
else:
    print(f"✅ Directory '{output_dir}' already exists.")

In [ ]:
# ✅ 문제 1 정답
personas = {
    "초보": {
        "description": "투자 경험이 거의 없고, 원금 보존을 최우선시하는 투자자",
        "queries": [
            {"query": "안전한 투자 상품 추천해주세요", "expected_category": ["리츠", "ESG"]}, # Updated
            {"query": "원금 손실 위험 없는 ETF", "expected_category": ["리츠"]}, # Updated
            {"query": "적금보다 나은 안전한 투자", "expected_category": ["리츠", "원자재"]}, # Updated
        ]
    },
    "중급": {
        "description": "기본적인 투자 지식이 있고, 적절한 위험을 감수하며 자산 증식을 추구하는 투자자",
        "queries": [
            {"query": "성장성과 안정성을 모두 고려한 ETF", "expected_category": ["ESG", "헬스케어"]}, # New Query
            {"query": "2차전지 산업 성장과 관련된 ETF", "expected_category": ["2차전지"]}, # New Query
            {"query": "인플레이션 헤지를 위한 포트폴리오", "expected_category": ["원자재", "리츠"]}, # New Query
        ]
    },
    "전문": {
        "description": "투자 경험이 풍부하고, 높은 수익을 위해 높은 변동성을 감수할 수 있는 투자자",
        "queries": [
            {"query": "신기술 섹터 ETF 심층 분석", "expected_category": ["2차전지", "헬스케어"]}, # Updated
            {"query": "고위험 고수익 노릴 수 있는 ETF", "expected_category": ["2차전지"]}, # Updated
            {"query": "시장 변동성 활용한 투자 전략", "expected_category": ["원자재"]}, # Updated
        ]
    },
}

with open("project2_data/query_set.json", "w") as f:
    json.dump(personas, f, ensure_ascii=False, indent=2)

print(f"✅ {sum(len(p['queries']) for p in personas.values())}개 질의 저장 완료")

In [ ]:
import os
import json

# Define the personas dictionary (assuming it's already defined or loaded in context)
personas = {
    "초보": {
        "description": "투자 경험이 거의 없고, 원금 보존을 최우선시하는 투자자",
        "queries": [
            {"query": "안전한 투자 상품 추천해주세요", "expected_category": ["리츠", "ESG"]}, # Updated
            {"query": "원금 손실 위험 없는 ETF", "expected_category": ["리츠"]}, # Updated
            {"query": "적금보다 나은 안전한 투자", "expected_category": ["리츠", "원자재"]}, # Updated
        ]
    },
    "중급": {
        "description": "기본적인 투자 지식이 있고, 적절한 위험을 감수하며 자산 증식을 추구하는 투자자",
        "queries": [
            {"query": "성장성과 안정성을 모두 고려한 ETF", "expected_category": ["ESG", "헬스케어"]}, # New Query
            {"query": "2차전지 산업 성장과 관련된 ETF", "expected_category": ["2차전지"]}, # New Query
            {"query": "인플레이션 헤지를 위한 포트폴리오", "expected_category": ["원자재", "리츠"]}, # New Query
        ]
    },
    "전문": {
        "description": "투자 경험이 풍부하고, 높은 수익을 위해 높은 변동성을 감수할 수 있는 투자자",
        "queries": [
            {"query": "신기술 섹터 ETF 심층 분석", "expected_category": ["2차전지", "헬스케어"]}, # Updated
            {"query": "고위험 고수익 노릴 수 있는 ETF", "expected_category": ["2차전지"]}, # Updated
            {"query": "시장 변동성 활용한 투자 전략", "expected_category": ["원자재"]}, # Updated
        ]
    }
}

output_dir = "project2_data"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"✅ Directory '{output_dir}' created.")
else:
    print(f"✅ Directory '{output_dir}' already exists.")

with open(os.path.join(output_dir, "query_set.json"), "w") as f:
    json.dump(personas, f, ensure_ascii=False, indent=2)

print(f"✅ {sum(len(p['queries']) for p in personas.values())}개 질의 저장 완료")

# 문제 2: 추가 ETF 문서 생성
5개 추가 ETF(ESG, 2차 전지, 헬스케어, 리츠, 원자재)를 정의하고 LLM으로 설명을 생성하세요.

요구사항:
    - additional_etfs 리스트: 5개 ETF (name, category, market)
    - risk_map 딕셔너리: 카테고리별 위험도 매핑
    - 각 ETF에 대해 client.chat.completions.create()로 설명 생성
    - etf_documents에 추가 후 카테고리별 통계 출력

In [ ]:
# ✅ 문제 2 정답
additional_etfs = [
    {"name": "TIGER ESG리더스", "category": "ESG", "market": "국내"},
    {"name": "KODEX 2차전지산업", "category": "2차전지", "market": "국내"},
    {"name": "TIGER 헬스케어", "category": "헬스케어", "market": "국내"},
    {"name": "KODEX 한국부동산리츠인프라", "category": "리츠", "market": "국내"},
    {"name": "KODEX 골드선물", "category": "원자재", "market": "글로벌"},
]

risk_map = {
    "ESG": 3, "2차전지": 4, "헬스케어": 3, "리츠": 2, "원자재": 3,
    "인덱스": 2, "섹터": 4, "배당": 2, "레버리지": 5, "테마": 4, "채권": 1, "자산배분": 2
}

etf_documents = [] # <-- Add this line to initialize the list

for etf in additional_etfs:
    prompt = f"""다음 ETF에 대한 상세 설명을 작성해주세요:
    - ETF명: {etf['name']}
    - 카테고리: {etf['category']}
    - 시장: {etf['market']}

    다음 항목을 포함해주세요:
    1. 투자 전략 (3-4문장)
    2. 주요 편입 종목 (5개)
    3. 수수료 및 비용 (총보수)
    4. 적합한 투자자 유형
    5. 주의사항

    한국어로 300-400자 내외로 작성해주세요."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

    doc_text = response.choices[0].message.content
    etf_documents.append({
        "name": etf["name"],
        "category": etf["category"],
        "market": etf["market"],
        "content": doc_text
    })
    print(f"✅ {etf['name']} 문서 생성 완료 (위험도: {risk_map.get(etf['category'], '?')})")

df = pd.DataFrame(etf_documents)
print(f"\n카테고리별 수:\n{df['category'].value_counts()}")

In [ ]:
import uuid # uuid 모듈을 import 합니다.

# ETF 문서들을 LangChain Document 형식으로 변환
etf_docs = []
for etf in etf_documents:
    doc_id = str(uuid.uuid4()) # Generate a UUID
    metadata = {
        "name": etf["name"],
        "category": etf["category"],
        "market": etf["market"],
        "doc_id": doc_id # Assign the generated UUID to 'doc_id' in metadata
    }
    etf_docs.append(Document(page_content=etf["content"], metadata=metadata, id=doc_id)) # Also set the 'id' of the Document object

print(f"✅ {len(etf_docs)}개의 ETF 문서가 LangChain Document 형식으로 변환되었습니다.")

In [ ]:
from langchain_community.vectorstores import FAISS

# FAISS 벡터 스토어 초기화 및 문서 색인
vectorstore = FAISS.from_documents(etf_docs, embeddings)
print("✅ FAISS 벡터 스토어 초기화 및 문서 색인 완료")

In [ ]:
etf_docs

# 문제 3: 멀티-관련성 질의 생성
하나의 질의에 여러 ETF가 관련되는 멀티-관련성 질의를 생성하세요.

### 요구사항:
    - multi_queries 리스트: 각 항목에 query, relevant(관련문서 리스트), irrelevant 포함
    - cats 변수에 카테고리 분포 저장
    - matplotlib으로 카테고리 바 차트 시각화
    - csv 파일로 저장

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os

# 폰트 설정 (한글 깨짐 방지)
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

import matplotlib.font_manager as fm

# fm._rebuild() # 이 줄은 일부 matplotlib 버전에서 오류를 발생시킬 수 있어 제거합니다.

plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False # 마이너스 폰트 깨짐 방지


multi_queries = [
    {
        "query": "분산투자에 좋은 안전한 포트폴리오 구성 추천",
        "relevant": [{"doc_id": 0, "score": 2}, {"doc_id": 8, "score": 3}, {"doc_id": 9, "score": 2}],
        "irrelevant": [6]
    },
    {
        "query": "미국 시장에 투자할 수 있는 ETF 비교",
        "relevant": [{"doc_id": 1, "score": 3}, {"doc_id": 2, "score": 3}, {"doc_id": 5, "score": 2}],
        "irrelevant": [3, 8]
    },
    {
        "query": "배당 수익과 안정성을 동시에 추구하는 ETF",
        "relevant": [{"doc_id": 4, "score": 3}, {"doc_id": 5, "score": 2}, {"doc_id": 8, "score": 2}],
        "irrelevant": [2, 6]
    },
]

# eval_dataset을 personas의 질의 목록에서 재구성합니다.
eval_dataset = []
for persona_type, persona_data in personas.items():
    for query_data in persona_data['queries']:
        eval_dataset.append(query_data)

# 카테고리 분포 시각화
# eval_dataset의 'expected_category' 리스트에서 모든 카테고리를 추출합니다.
all_categories = []
for item in eval_dataset:
    all_categories.extend(item['expected_category'])

cats = all_categories # 모든 개별 카테고리 목록
cat_counts = pd.Series(cats).value_counts()

plt.figure(figsize=(8, 4))
cat_counts.plot(kind='bar')
plt.title('평가 질의 카테고리 분포')
plt.xlabel('카테고리')
plt.ylabel('질의 수')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 평가 데이터 저장 디렉토리를 생성합니다.
output_eval_dir = "project2_data/evaluation"
if not os.path.exists(output_eval_dir):
    os.makedirs(output_eval_dir)
    print(f"✅ Directory '{output_eval_dir}' created.")
else:
    print(f"✅ Directory '{output_eval_dir}' already exists.")

# CSV 저장
pd.DataFrame(eval_dataset).to_csv(os.path.join(output_eval_dir, "eval_queries.csv"), index=False)
print("✅ CSV 저장 완료")

In [ ]:
# etf_docs의 카테고리와 문서 ID를 매핑합니다.
category_to_doc_ids = {}
for doc in etf_docs:
    category = doc.metadata['category']
    doc_id = doc.id
    if category not in category_to_doc_ids:
        category_to_doc_ids[category] = []
    category_to_doc_ids[category].append(doc_id)

# eval_dataset의 각 질의에 relevant_doc_ids를 추가합니다.
for item in eval_dataset:
    item['relevant_doc_ids'] = []
    for expected_cat in item['expected_category']:
        if expected_cat in category_to_doc_ids:
            item['relevant_doc_ids'].extend(category_to_doc_ids[expected_cat])
    # 중복 ID 제거 (하나의 문서가 여러 예상 카테고리에 속할 경우)
    item['relevant_doc_ids'] = list(set(item['relevant_doc_ids']))

print("✅ eval_dataset에 relevant_doc_ids 추가 완료")

위 셀을 실행한 후, 다시 문제 5의 평가 지표 계산 셀을 실행하시면 정상적으로 작동할 것입니다.

# 문제 4: MMR 검색 구현
벡터 스토어에서 MMR(Maximal Marginal Relevance) 검색을 구현하고 유사도 검색과 비교하세요.

### 요구사항:
    - format_results(results, method_name) 함수: 검색 결과를 포맷팅하여 출력
    - Smilarity Search와 MMR Search 각각 실행 후 시간 측정
    - k 값(3, 5, 10)별 카테고리 다양성 비교

In [ ]:
# ✅ 문제 4 정답
import time

def format_results(results, method_name):
    print(f"\n🔍 {method_name} 결과:")
    for i, item in enumerate(results, 1):
        if isinstance(item, tuple) and len(item) == 2:
            doc, score = item
            print(f"  {i}. [{score:.4f}] {doc.metadata['name']} ({doc.metadata['category']})")
        else:
            print(f"  {i}. {item.metadata['name']} ({item.metadata['category']})")

query = "분산 투자에 적합한 ETF"

# Similarity Search
start = time.time()
sim_results = vectorstore.similarity_search_with_score(query, k=5)
sim_time = time.time() - start
format_results(sim_results, f"Similarity Search ({sim_time:.3f}s)")

# MMR Search
start = time.time()
mmr_results = vectorstore.max_marginal_relevance_search(query, k=5, fetch_k=10)
mmr_time = time.time() - start
format_results([(doc, 0) for doc in mmr_results], f"MMR Search ({mmr_time:.3f}s)")

# k값별 카테고리 다양성 비교
print("\n📊 k값별 카테고리 다양성:")
for k in [3, 5, 10]:
    sim = vectorstore.similarity_search(query, k=k)
    mmr = vectorstore.max_marginal_relevance_search(query, k=k, fetch_k=max(k*2, 10))
    sim_cats = set(d.metadata['category'] for d in sim)
    mmr_cats = set(d.metadata['category'] for d in mmr)
    print(f"  k={k}: Similarity {len(sim_cats)}개 카테고리 vs MMR {len(mmr_cats)}개 카테고리")

In [ ]:
from rank_bm25 import BM25Okapi

# ETF 문서들의 텍스트 콘텐츠를 토큰화
tokenized_corpus = [doc.page_content.split(" ") for doc in etf_docs]

bm25 = BM25Okapi(tokenized_corpus)

# BM25 검색 함수 정의 (스코어 반환하도록 수정)
def bm25_search(query, k=5):
    tokenized_query = query.split(" ")
    doc_scores = bm25.get_scores(tokenized_query)
    top_n_indices = np.argsort(doc_scores)[::-1][:k]
    results_with_scores = []
    for i in top_n_indices:
        results_with_scores.append((etf_docs[i], doc_scores[i])) # 문서와 스코어 함께 반환
    return results_with_scores

print("✅ BM25 검색 기능이 구현되었습니다. (스코어 반환 기능 추가)")

# 문제 5: Precison@K와 Recall@K 구현

### Precision@K 와 Recall@K 평가 지표를 구현하세요.

요구사항:
    - precision_at_k(vectorstore, eval_data, k=5) 함수 구현
    - recall_at_k(vectorstore, eval_data, k=5) 함수 구현
    - k=1~10에서 Hit rate, MRR, Precision, Recall 4개 지표를 그래프로 비교

In [ ]:
import matplotlib.pyplot as plt

# 문제 5: Precision@K, Recall@K 구현
def precision_at_k(vectorstore, eval_dataset, k=5):
    precisions=[]
    for item in eval_dataset:
        results = vectorstore.similarity_search(item['query'], k=k) # vectorsearch의 유사 문서 결과
        retrieved_ids = [r.id for r in results] # 유사문서 검색결과를 top-k의 문서 id 추출
        relevant_retrieved = sum(1 for rid in retrieved_ids if rid in item['relevant_doc_ids']) #

        precisions.append(relevant_retrieved / k)

    return np.mean(precisions)


def recall_at_k(vectorstore, eval_dataset, k=5):
    recalls = []
    for item in eval_dataset:
        results = vectorstore.similarity_search(item['query'], k=k)
        retrieved_ids = [r.id for r in results]
        relevant_retrieved = sum(1 for rid in retrieved_ids if rid in item['relevant_doc_ids'])
        total_relevant = len(item['relevant_doc_ids'])

        recalls.append(relevant_retrieved / total_relevant if total_relevant > 0 else 0)

    return np.mean(recalls)

def hit_rate_at_k(vectorstore, eval_dataset, k=5):
    hits = []
    for item in eval_dataset:
        results = vectorstore.similarity_search(item['query'], k=k)
        retrieved_ids = [r.id for r in results]
        # 한 개라도 관련 문서가 검색되면 Hit
        if any(rid in item['relevant_doc_ids'] for rid in retrieved_ids):
            hits.append(1)
        else:
            hits.append(0)
    return np.mean(hits)

def mrr_at_k(vectorstore, eval_dataset, k=5):
    mrrs = []
    for item in eval_dataset:
        results = vectorstore.similarity_search(item['query'], k=k)
        retrieved_ids = [r.id for r in results]

        # 첫 번째 관련 문서의 순위를 찾습니다.
        rank = 0
        for i, rid in enumerate(retrieved_ids):
            if rid in item['relevant_doc_ids']:
                rank = i + 1  # 1-based rank
                break

        if rank > 0:
            mrrs.append(1 / rank)
        else:
            mrrs.append(0)
    return np.mean(mrrs)

ks = range(1, 11)
# ---- Hit Rate, MRR, Precision, Recall 4개 비교 차트 ---- #
hrs = [hit_rate_at_k(vectorstore, eval_dataset, k) for k in ks]
mrrs = [mrr_at_k(vectorstore, eval_dataset, k) for k in ks]
precs = [precision_at_k(vectorstore, eval_dataset, k) for k in ks]
recs = [recall_at_k(vectorstore, eval_dataset, k) for k in ks]

plt.figure(figsize=(10, 6))
plt.plot(ks, hrs, 'o-', label='Hit Rate')
plt.plot(ks, mrrs, 's-', label='MRR')
plt.plot(ks, precs, '^-', label='Precision')
plt.plot(ks, recs, 'D-', label='Recall')
plt.xlabel('K')
plt.ylabel('Score')
plt.title('검색 평가 지표 비교 (K=1~10)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np

def dcg_at_k(relevances, k):
    relevances = np.asarray(relevances)[:k]
    if relevances.size == 0 or np.sum(relevances) == 0: # Check if relevances is empty or all zeros
        return 0.0
    return np.sum(relevances / np.log2(np.arange(2, relevances.size + 2)))

def ndcg_at_k(relevances, k):
    dcg_max = dcg_at_k(sorted(relevances, reverse=True), k)
    if not dcg_max:
        return 0.0
    return dcg_at_k(relevances, k) / dcg_max

print("✅ DCG 및 NDCG 함수 구현 완료")

## 현재 문제 진단을 위한 추가 확인

평가 지표가 모두 0으로 나오는 것은 `similarity_search`가 관련 문서를 전혀 찾지 못하고 있음을 의미합니다. 이를 확인하기 위해 `eval_dataset`의 `relevant_doc_ids` 내용을 살펴보고, 특정 질의에 대한 `similarity_search` 결과를 직접 확인해 보겠습니다.

In [ ]:
# eval_dataset의 샘플을 확인하여 relevant_doc_ids가 제대로 채워졌는지 확인
print("✅ eval_dataset 샘플 확인:")
for i, item in enumerate(eval_dataset):
    print(f"Query: {item['query']}")
    print(f"Expected Categories: {item['expected_category']}")
    print(f"Relevant Doc IDs: {item['relevant_doc_ids']}")
    if i >= 2: # 상위 3개만 확인
        break

In [ ]:
# 첫 번째 질의에 대한 similarity_search 결과와 relevant_doc_ids 비교
sample_query_item = eval_dataset[0]
sample_query_text = sample_query_item['query']
sample_relevant_ids = sample_query_item['relevant_doc_ids']

print(f"\n🔍 샘플 질의: {sample_query_text}")
print(f"  예상 관련 문서 ID: {sample_relevant_ids}")

sample_retrieved_docs = vectorstore.similarity_search(sample_query_text, k=5)
retrieved_ids = [doc.id for doc in sample_retrieved_docs]

print(f"  검색 결과 (top-5) 문서 ID:")
for doc in sample_retrieved_docs:
    print(f"    - {doc.id} (카테고리: {doc.metadata['category']}, 이름: {doc.metadata['name']})")

# 검색된 문서 중 관련 문서가 있는지 확인
found_relevant = [rid for rid in retrieved_ids if rid in sample_relevant_ids]
print(f"  ✅ 검색 결과 중 관련 문서: {found_relevant}")

if not found_relevant:
    print("  ❗️ 검색된 문서 중에 예상 관련 문서가 없습니다. 검색 성능에 문제가 있을 수 있습니다.")
else:
    print("  🎉 검색된 문서 중에 관련 문서가 있습니다.")

# 문제 6: NDCG 평가 리포트

Graded relevance(0, 1, 2, 3)를 사용한 NDCG 평가 리포트를 작성하세요.

요구사항:
    - report 딕셔너리에 k=1, 3, 5, 10 별 Hit Rate, MRR, NDCG 저장
    - JSON 파일로 저장 (baseline_report.json)
    - 결과를 표 형태로 출력

In [ ]:
report = {'baseline' : {}}

for k in [1, 3, 5, 10]:
    hr = hit_rate_at_k(vectorstore, eval_dataset, k=k)
    mrr = mrr_at_k(vectorstore, eval_dataset, k=k)

    ndcg_scores = []
    for item in eval_dataset:
        results = vectorstore.similarity_search(item['query'], k=k)
        rels = [1 if r.id in item["relevant_doc_ids"] else 0 for r in results]
        ndcg_scores.append(ndcg_at_k(rels, k))


    report['baseline'][f'k={k}'] = {
        'hit_rate': round(hr, 4),
        'mrr': round(mrr, 4),
        'ndcg': round(np.mean(ndcg_scores), 4)
    }

with open('project2_data/evaluation/baseline_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f"{'K':<6} | {'Hit Rate':>10} | {'MRR':>10} | {'NDCG':>10}")
print("-" * 45)
for k_str, metrics in report['baseline'].items():
    print(f"{k_str:<6} | {metrics['hit_rate']:>10.4f} | {metrics['mrr']:>10.4f} | {metrics['ndcg']:>10.4f}")

print(f"\n✅ baseline_report.json 저장 완료")

# 문제 7: BM25 vs FAISS 비교평가

BM25 검색 결과를 FAISS 검색 결과와 비교 평가하세요.

요구사항:
    - bm25_hit_rate(eval_data, k=5) 함수 구현
    - bm25_mrr(eval_data, k=5) 함수 구현
    - BM25와 FAISS의 Hit Rate, MRR 비교 테이블 출력

In [ ]:
def bm25_hit_rate(eval_dataset, k=5):
    hits = 0
    for item in eval_dataset:
        results = bm25_search(item['query'], k)
        retrieved_ids = [r[0].id for r in results]
        if any(rid in item['relevant_doc_ids'] for rid in retrieved_ids):
            hits += 1
    return hits / len(eval_dataset)

def bm25_mrr(eval_dataset, k=5):
    rr_sum = 0
    for item in eval_dataset:
        results = bm25_search(item['query'], k)
        # results는 LangChain Document 객체이므로 metadata['doc_id']가 아닌 id 속성을 사용합니다.
        retrieved_ids = [r[0].id for r in results]

        for rank, rid in enumerate(retrieved_ids, 1):
            if rid in item['relevant_doc_ids']:
                rr_sum += 1 / rank
                break
    return rr_sum / len(eval_dataset)

print(f"{'방법':<12} | {'K':>3} | {'Hit Rate':>10} | {'MRR':>10}")
print("-" * 45)
for k in [1, 3, 5, 10]:
    faiss_hr = hit_rate_at_k(vectorstore, eval_dataset, k)
    faiss_mrr = mrr_at_k(vectorstore, eval_dataset, k)
    bm25_hr = bm25_hit_rate(eval_dataset, k)
    bm25_m = bm25_mrr(eval_dataset, k)
    print(f"{'FAISS':<12} | {k:>3} | {faiss_hr:>10.4f} | {faiss_mrr:>10.4f}")
    print(f"{'BM25':<12} | {k:>3} | {bm25_hr:>10.4f} | {bm25_m:>10.4f}")
    print("-" * 45)

# 문제 8: Alpha 최적화

Alpha값을 0.0 ~ 1.0까지 0.1 단위로 변경하며 Hit Rate를 측정하고 최적값을 찾으세요

### 요구사항:
    - alpha를 0.0 ~ 1.0 범위에서 0.1 간격으로 탐색
    - 각 alpha별 Hit Rate@5 출력
    - 최적 alpha와 Hit Rate를 chechpoint 딕셔너리에 저장
    - project2_data/checkpoints/hybrid_config.json으로 저장

In [ ]:
def hybrid_search(query, alpha=0.5, k=5, fetch_k=20):
    # FAISS (Vector Store) 검색
    faiss_results_with_scores = vectorstore.similarity_search_with_score(query, k=fetch_k)

    # BM25 검색
    bm25_results_with_scores = bm25_search(query, k=fetch_k)

    # 결과 정규화 및 결합
    # doc_id를 키로, 최대 점수를 값으로 하는 딕셔너리 생성 (동일 문서에 대한 여러 점수 처리)
    combined_scores = {}

    # FAISS 점수 정규화 및 합산
    if faiss_results_with_scores:
        # LangChain의 similarity_search_with_score는 점수가 낮을수록 유사도가 높은 것으로 간주하므로, 반전하여 정규화합니다.
        # 즉, 점수가 낮을수록 (더 유사할수록) 높은 정규화된 점수를 갖도록 합니다.
        faiss_scores = [score for doc, score in faiss_results_with_scores]
        if not faiss_scores: # 리스트가 비어있으면 건너뛰기
            pass
        else:
            max_faiss_score = max(faiss_scores)
            min_faiss_score = min(faiss_scores)
            if max_faiss_score == min_faiss_score: # 모든 점수가 동일한 경우 0.5로 정규화
                normalized_faiss_scores = {doc.id: 0.5 for doc, score in faiss_results_with_scores}
            else:
                # 점수가 낮을수록 유사도가 높으므로, (max_score - score) / (max_score - min_score) 로 정규화
                normalized_faiss_scores = {
                    doc.id: (max_faiss_score - score) / (max_faiss_score - min_faiss_score)
                    for doc, score in faiss_results_with_scores
                }
            for doc, score in faiss_results_with_scores:
                combined_scores[doc.id] = combined_scores.get(doc.id, 0) + alpha * normalized_faiss_scores[doc.id]

    # BM25 점수 정규화 및 합산
    if bm25_results_with_scores:
        bm25_scores = [score for doc, score in bm25_results_with_scores]
        if not bm25_scores: # 리스트가 비어있으면 건너뛰기
            pass
        else:
            max_bm25_score = max(bm25_scores)
            min_bm25_score = min(bm25_scores)
            if max_bm25_score == min_bm25_score: # 모든 점수가 동일한 경우 0.5로 정규화
                normalized_bm25_scores = {doc.id: 0.5 for doc, score in bm25_results_with_scores}
            else:
                # BM25는 점수가 높을수록 유사도가 높으므로, (score - min_score) / (max_score - min_score) 로 정규화
                normalized_bm25_scores = {
                    doc.id: (score - min_bm25_score) / (max_bm25_score - min_bm25_score)
                    for doc, score in bm25_results_with_scores
                }
            for doc, score in bm25_results_with_scores:
                combined_scores[doc.id] = combined_scores.get(doc.id, 0) + (1 - alpha) * normalized_bm25_scores[doc.id]

    # 합산된 점수를 기준으로 정렬
    sorted_results = sorted(combined_scores.items(), key=lambda item: item[1], reverse=True)

    # 원본 문서 객체와 매핑
    final_results_with_scores = [] # (Document 객체, 최종 점수) 튜플 리스트로 반환하도록 수정
    doc_id_map = {doc.id: doc for doc in etf_docs} # 문서 ID로 문서 객체 찾기 위한 맵
    for doc_id, score in sorted_results:
        if doc_id in doc_id_map:
            final_results_with_scores.append((doc_id_map[doc_id], score)) # (Document 객체, 점수) 튜플 추가
        if len(final_results_with_scores) == k:
            break

    return final_results_with_scores

best_alpha, best_hr = 0, 0

print(f"{'Alpha':>6} | {'Hit Rate@5':>12}")
print("-" * 25)

# project2_data/checkpoints 디렉토리가 없으면 생성
output_checkpoint_dir = "project2_data/checkpoints"
if not os.path.exists(output_checkpoint_dir):
    os.makedirs(output_checkpoint_dir)
    print(f"✅ Directory '{output_checkpoint_dir}' created.")

for a in np.arange(0, 1.1, 0.1):
    a = round(a, 1)
    hits = 0
    for item in eval_dataset:
        # hybrid_search가 (Document, score) 튜플 리스트를 반환하도록 변경되었으므로, r은 (Document, score) 튜플임
        results_with_scores = hybrid_search(item['query'], alpha=a, k=5)
        retrieved_ids = [doc.id for doc, score in results_with_scores] # Document 객체에서 id 추출
        if any(rid in item['relevant_doc_ids'] for rid in retrieved_ids):
            hits += 1
    hr = hits / len(eval_dataset)
    print(f"{a:6.1f} | {hr:>12.4f}")
    if hr > best_hr:
        best_alpha, best_hr = a, hr

checkpoint = {'best_alpha': best_alpha, 'best_hr': round(best_hr, 4)}
with open(os.path.join(output_checkpoint_dir, "hybrid_config.json"), "w") as f:
    json.dump(checkpoint, f, ensure_ascii=False, indent=2)

print(f"\n🏆 최적 alpha: {best_alpha} (Hit Rate: {best_hr:.4f})")
print("✅ hybrid_config.json 저장 완료")

# 문제 9: 도메인 특화 동의어 사전

도메인 특화 동의어 사전을 만들고 쿼리 확장 성능을 비교하세요.

### 요구사항:
    - finance_synonyms 딕셔너리: ETF, 배당, 안정, 성장, 미국 등의 동의어
    - synonym_expan(query) 함수: 동의어로 확장된 쿼리 리스트 반환
    - Baseline vs Synonym 확장 Hit Rate 비교

In [ ]:

finance_synonyms = {
    'ETF': ['상장지수펀드', '인덱스펀드', '지수추종'],
    '배당': ['분배금', '배당금', '인컴', '이자수익'],
    '안정': ['안전', '보수적', '저위험', '원금보존'],
    '성장': ['그로스', '공격적', '고수익', '고성장'],
    '미국': ['해외', '글로벌', '나스닥', 'S&P'],
}

def synonym_expand(query):
    expanded = [query]
    for keyword, synonyms in finance_synonyms.items():
        if keyword in query:
            for syn in synonyms:
                # 원본 쿼리의 키워드를 동의어로 대체하여 추가
                expanded.append(query.replace(keyword, syn))
    return expanded

# Baseline vs Synonym Hit Rate 비교
baseline_hits, synonym_hits = 0, 0

for item in eval_dataset:
    # Baseline
    # hybrid_search는 이제 (Document, score) 튜플 리스트를 반환합니다.
    results_with_scores = hybrid_search(item['query'], alpha=0.5, k=5)
    retrieved_docs_baseline = [doc for doc, score in results_with_scores] # Document 객체만 추출

    if any(r.id in item['relevant_doc_ids'] for r in retrieved_docs_baseline):
        baseline_hits += 1

    # Synonym expanded
    expanded_queries = synonym_expand(item['query'])
    all_results = {}

    for eq in expanded_queries:
        # hybrid_search는 이제 (Document, score) 튜플 리스트를 반환합니다.
        current_results_with_scores = hybrid_search(eq, alpha=0.5, k=5)
        for doc, score in current_results_with_scores: # doc은 Document 객체, score는 해당 문서의 점수
            doc_id = doc.id
            # 동의어 확장 쿼리에서 동일한 doc_id에 대해 더 높은 점수가 나오면 업데이트
            if doc_id not in all_results or score > all_results[doc_id][1]:
                all_results[doc_id] = (doc, score) # (Document 객체, 점수) 형태로 저장

    # 전체 확장 쿼리에서 가장 높은 점수의 상위 5개 문서 추출
    # all_results의 values는 (Document, score) 튜플이므로, score (x[1])를 기준으로 정렬
    top_results_with_scores = sorted(all_results.values(), key=lambda x: x[1], reverse=True)[:5]

    # top_results_with_scores는 [(Document, score), ...] 형태이므로, Document 객체만 추출
    retrieved_docs_synonym = [doc for doc, score in top_results_with_scores]

    if any(r.id in item['relevant_doc_ids'] for r in retrieved_docs_synonym):
        synonym_hits += 1

print(f"Baseline Hit Rate: {baseline_hits/len(eval_dataset):.4f}")
print(f"Synonym  Hit Rate: {synonym_hits/len(eval_dataset):.4f}")

# 문제 10: 커스텀 Multi-Query Retriever

커스텀 프롬프트로 Multi-Query Retriever를 설정하세요.

### 요구사항:
    - custom_prompt PromptTemplate 정의 (ETF 검색 전문가 역할)
    - retriever_custom = MultiQueryRetriever 생성
    - Retriver_custom.invoke()로 검색 실행 후 결과 출력
    

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain.retrievers.multi_query import MultiQueryRetriever



custom_prompt = PromptTemplate(
    input_variables=['question'],
    template="""당신은 ETF 금융 상품 검색 전문가입니다.
다음 질문을 서로 다른 관점에서 3가지로 재작성하세요.
각 질의는 ETF 검색에 최적화되어야 합니다.

원래 질문: {question}

재작성된 질의 (한 줄에 하나씩):"""
)

retriever_custom = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    llm=llm,
    prompt=custom_prompt
)

results_custom = retriever_custom.invoke("노후 대비 안정적 투자")
print(f"🔍 커스텀 Multi-Query 결과: {len(results_custom)}개")
for doc in results_custom:
    print(f"  - {doc.metadata['name']}: {doc.page_content[:80]}...")

# 문제 11: 검색 비교 대시보드

모든 검색 방법의 결과를 나란히 비교하는 Gradio UI를 만드세요.

### 요구사항:
    - full_comparison(query, top_k) 함수: FAISS/BM25/Hybrid 결과 비교
    - show_history() 함수: 최근 검색 이력 표시
    - gr.Blocks로 탭UI 구성 (검색 탭 + 이력 탭)

In [ ]:
import gradio as gr

search_history = []

def full_comparison(query, top_k):
    top_k = int(top_k)

    output = f"질의 : {query}\n{'='*60}\n\n"

    # FAISS
    faiss_results = vectorstore.similarity_search_with_score(query, k=top_k)
    output += "FAISS 벡터 검색:\n"
    for i, (doc, score) in enumerate(faiss_results, 1):
        output += f" {i}. [{score:.4f}] {doc.metadata['name']} ({doc.metadata['category']})\n"

    # BM25
    bm25_results = bm25_search(query, top_k)
    output += "\n BM25 키워드 검색:\n"
    for i, r in enumerate(bm25_results, 1):
        output += f" {i}. [{r['score']:.4f}] {r['name']}\n"

    # Hybrid
    hybrid_results = hybrid_search(query, alpha=0.5, k=top_k)
    output += "\n Hybrid 검색:\n"
    for i, (doc_id, score, name) in enumerate(hybrid_search_results, 1):
        output += f" {i}. [{score:.4f}] {name}\n"

    return output

def show_history():
    if not search_history:
        return "검색 이력이 없습니다."
    return "\n".join(f"{i+1}. {q}" for i, q in enumerate(search_history))

with gr.Blocks(title="ETF 검색 비교 대시보드") as demo:
    with gr.Tab("검색"):
        query_input = gr.Textbox(label="검색 질의", placeholder="예: 배당 수익률 높은 안전한 ETF")
        top_k_slider = gr.Slider(1, 10, value=5, step=1, label="결과 수 (K)")
        search_btn = gr.Button("검색")
        output = gr.Textbox(label="비교 결과", lines=20)
        search_btn.click(full_comparison, [query_input, top_k_slider], output)

        with gr.Tab("검색 이력"):
            history_btn = gr.Button("이력 조회")
            history_output = gr.Textbox(label="최근 검색 이력", lines=10)
            history_btn.click(show_history, [], history_output)

    demo.launch(share=True)